# Results Walkthrough — *Does showing a language model the downside change the reward code it writes?*

**A reproducible, end-to-end analysis of the LLM-reward-design experiment.**

An LLM authors the *reward-function code* for a fixed risk-sensitive deep-RL portfolio agent (SB3 SAC) trading the point-in-time top-30 of a survivorship-free S&P 500 panel. Across arms we vary **only the feedback channel** — a multi-level **left-tail vector** (six coordinates) versus a **scalar** performance number — and ask whether the richer signal produces better risk-adjusted, tail-aware policies. The pre-registered prediction is a **bounded null**, reported not as a failure but as a **mechanism boundary condition**: the headline is the *mechanism* — fed signal → authored code → policy → realised tail — and a null **locates where the chain breaks**.

> **Status.** The confirmatory campaign is unrun, so every *inferential* number below is rendered on a **synthetic NULL-shaped demo** (`make_figures.synthesize_null`) that exercises the real analysis engine. *No inferential number here is a result.* The data, EDA, and taxonomy sections **are real** (frozen gold panel + the prototype reward archive). Post-campaign, replace the one `synthesize_null(...)` call with the sealed-leg loader and every cell re-runs **identically** — the notebook is the analysis contract.

## Reproducibility contract

- **This notebook is generated.** `scripts/build_notebook_results.py` authors every cell deterministically (fixed cell ids, no timestamps); the shipped copy stores **no outputs**. Regenerate with `python scripts/build_notebook_results.py`; validate by executing top-to-bottom (`jupyter nbconvert --to notebook --execute`).
- **Determinism.** Every stochastic cell below draws from an explicitly seeded `numpy` generator, so re-execution replays byte-identically. LLM calls are **non-deterministic and are replayed from the archive**, never regenerated (hosted frontier APIs removed `temperature`/`seed`).
- **Provenance.** The design is pinned by `scripts/freeze.py` (SHA-256 over the prereg + prompts + `arms.yaml` + the inference family + `config/data.yaml`, which binds the PANEL identity); §1 prints the live freeze state, and the companion `data_provenance_walkthrough.ipynb` re-verifies the data artifacts hash-by-hash.
- **Honest nulls.** Equivalence is read off **TOST vs the ±SESOI band** and **Bayes factors / a Model Confidence Set** — never off a bare *p* > 0.05.

## 0 · Setup

In [ ]:
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd

# Resolve the repo root whether the notebook is opened from notebooks/ or the repo root.
ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
for p in (str(ROOT), str(ROOT / 'scripts')):
    if p not in sys.path:
        sys.path.insert(0, p)

import matplotlib.pyplot as plt  # noqa: F401  (inline rendering; headless runs use Agg)
from src.viz.style import apply_house_style
apply_house_style()

RNG_SEED = 7
# Analysis-only stack ON PURPOSE: no torch / SB3 / CUDA is imported anywhere in this
# notebook — it reads frozen artifacts and runs seeded CPU statistics, so it can never
# contend with (or depend on) a live training run.
print('repo root :', ROOT)
print('python    :', platform.python_version())
print('numpy     :', np.__version__)
print('pandas    :', pd.__version__)

## 1 · Provenance & freeze state

What pins this analysis. Two facts are printed live rather than asserted from memory:

1. **The panel identity.** `src.data.loaders.gold_suffix()` resolves the active gold suffix from `config/data.yaml` (`gold.suffix`), which is **bound into the freeze hash** — the headline panel cannot be swapped silently. The active panel is `univ5` (Split C, ADR-044/051).
2. **The freeze state.** A frozen design (a non-null `freeze_hash`) is what licenses confirmatory claims; before the freeze this cell reports `frozen: False` — **by design**, and every claim in this notebook stays methodological until it flips.

In [ ]:
import yaml
from src.data.loaders import gold_suffix

suffix = gold_suffix()
assert suffix == 'univ5', f'active gold suffix is {suffix!r}, expected univ5 (Split C)'
prereg = yaml.safe_load((ROOT / 'config' / 'preregistration.yaml').read_text(encoding='utf-8'))
arms = list(prereg['arms'])
assert len(arms) == 7, f'expected the 7-arm frozen roster, found {len(arms)}'
prov = {
    'gold suffix (hash-bound)': suffix,
    'frozen': prereg.get('frozen'),
    'freeze_hash': prereg.get('freeze_hash'),
    'arms (n=7)': ', '.join(arms),
}
for k, v in prov.items():
    print(f'{k:>25} : {v}')

## 2 · The data — Split C on the `univ5` panel

**Why this panel.** The gold panel is a licensed Refinitiv/LSEG build: survivorship-free, point-in-time S&P 500 membership reconstructed by reverse event replay, daily **total** returns (dividends and terminal delisting returns included), 2005-01-03 → 2026-06-30. Dead names stay in the panel (Wachovia, Lehman-era casualties, Dell's 2013 take-private …) — dropping them would manufacture exactly the survivorship bias a *tail-risk* study cannot afford. The agent trades the **point-in-time top-30 megacap book**: the 30 largest names by market cap *known at the window start*, never today's list projected backwards.

**Split C** (ADR-044, executed 2026-07-02 with the ADR-051 rebuild):

| split | span | role |
|---|---|---|
| train | 2005-01-01 → 2016-12-31 | agent learns; feedback is *measured* here |
| validation | 2017-01-01 → 2019-12-31 | candidate selection (held-out Deflated Sharpe) |
| test | 2020-01-01 → 2026-06-30 | **sealed** until final inference |

The sealed test span deliberately contains the modern regime set — COVID (2020), the 2022 inflation-hiking drawdown, the 2023-25 AI rally, and settled H1-2026 — so the tail claim is evaluated where tails actually happened. Inter-split boundaries carry a purge of `max(embargo=21, feature lookback=60) = 60` sessions (López de Prado), so no observation window straddles a boundary.

In [ ]:
returns = pd.read_parquet(ROOT / 'data' / 'gold' / 'returns_panel_univ5.parquet')
top30 = pd.read_parquet(ROOT / 'data' / 'gold' / 'top30_selection_univ5.parquet')
assert returns.shape == (5406, 963), f'unexpected panel shape {returns.shape}'
assert str(returns.index[0])[:10] == '2005-01-03'
assert str(returns.index[-1])[:10] == '2026-06-30'
print(f'returns_panel_univ5 : {returns.shape[0]} sessions x {returns.shape[1]} RICs, '
      f'{str(returns.index[0])[:10]} -> {str(returns.index[-1])[:10]}')

# The point-in-time top-30 book at two window starts, 15 years apart. The lists are
# provenance-level metadata (they exist for audit); inside the experiment the panel is
# ANONYMISED to integer ids -- no RIC, ticker, or date ever reaches a reward or the LLM.
def book(phase, start):
    row = top30[(top30['phase'] == phase)
                & (pd.to_datetime(top30['window_start']) == pd.Timestamp(start))]
    assert len(row) == 1, f'no unique top-30 row for {phase} @ {start}'
    sel = list(row.iloc[0]['selection'])
    assert len(sel) == 30
    return sel

b2005 = book('development', '2005-01-03')
b2020 = book('walk_forward', '2020-01-02')
print('\ntop-30 @ 2005-01-03 :', ', '.join(b2005[:10]), '...')
print('top-30 @ 2020-01-02 :', ', '.join(b2020[:10]), '...')
same = sorted(set(b2005) & set(b2020))
print(f'overlap after 15y   : {len(same)}/30 names ({", ".join(same[:8])}...)')
print('turnover of the megacap book is itself a survivorship argument: '
      f'{30 - len(same)}/30 of the 2005 book is gone by 2020.')

### The design in one box

**Seven arms, one varying ingredient.** Every arm shares the identical environment, agent (SAC), search protocol, and budget; *only the feedback string shown to the reward-designing LLM differs* (the identification principle: nothing else may vary).

| arm | designer | feedback |
|---|---|---|
| `distributional` | LLM | six-coordinate left-tail vector (CVaR 1/5/10/25% + tail mass + robust skew) |
| `scalar` | LLM | one risk-adjusted scalar |
| `scalar_cvar5` | LLM | one tail scalar (CVaR-5%) |
| `placebo` | LLM | structurally matched, information-free string |
| `placebo_shuffled` | LLM | placebo with shuffled numbers (structure-vs-content control) |
| `random_search` | none | parameterised template, random constants |
| `bayes_opt` | none | parameterised template, BO-tuned constants |

**Headline test (H2).** Two **co-primary** one-sided equivalence questions, combined as intersection–union tests (Berger 1982) at SESOI $\Delta^\* = 0.05$:

$$H_0^{\mathrm{RA}}: |\,\overline{\mathrm{DSR}}_{\mathrm{dist}} - \overline{\mathrm{DSR}}_{\mathrm{scalar}}\,| \ge \Delta^\*
\qquad\text{vs}\qquad
H_1^{\mathrm{RA}}: |\Delta| < \Delta^\*$$

$$H_0^{\mathrm{Tail}}: |\,\mathrm{CVaR}^{5\%}_{\mathrm{dist}} - \mathrm{CVaR}^{5\%}_{\mathrm{scalar}}\,| \ge \Delta^\*
\qquad\text{vs}\qquad
H_1^{\mathrm{Tail}}: |\Delta| < \Delta^\*$$

The realized multiple-testing family is **enumerated and frozen at $m=6$**: 3 H2 contrasts (distributional vs {scalar, placebo, scalar_cvar5}) × 2 held-out metrics (Sharpe, CVaR-5%); Benjamini–Hochberg over the 6 is *reported*, never the gate — the IUT conjunction **is** the correction. Everything else in this notebook (taxonomy, mechanism kernel) is **report-only and disjoint** from that family: it can explain the headline, never gate it.

## 3 · EDA — the stylised facts that motivate the channel (real train window)

**Why this section exists.** The contribution is a *feedback channel*, so the design must be motivated by the **data the channel carries**. Before any model: does the training panel actually contain lower-tail structure that a single scalar cannot convey? Four classical stylised facts (Cont 2001), each recomputed **live** here on the **train window only** (development top-30, 2005-01-03 → 2016-12-30) — the sealed validation/test years are never read, so this EDA is snoop-clean by construction:

1. **(a) Heavy tails** — the crash days a matched Normal cannot see.
2. **(b) The tail is a curve, not a number** — empirical vs Normal-implied CVaR$_\alpha$ as $\alpha \to 0$, with the four *fed* levels marked.
3. **(c) Volatility clustering** — a time-averaged scalar hides *when* risk arrives.
4. **(d) Co-crashes** — diversification fails exactly in the tail.

The loader is called with `verify_checksum=True`: the panel's SHA-256 is re-verified against the frozen manifest **before** a single statistic is computed (the full hash-by-hash audit lives in `data_provenance_walkthrough.ipynb`).

In [ ]:
from src.data.loaders import load_gold_panel
from src.viz.eda import alive_mask_from_returns, fig_stylised_facts, stylised_fact_stats

dev = load_gold_panel('development', verify_checksum=True)  # TRAIN window only
R, dates = dev.panel.returns, dev.panel.dates
alive = alive_mask_from_returns(R)
stats = stylised_fact_stats(R, alive_mask=alive)
d0, d1 = str(dates[0])[:10], str(dates[-1])[:10]
assert (d0, d1) == ('2005-01-03', '2016-12-30'), f'train window drifted: {d0}..{d1}'

t3, t5 = stats['tail_3sigma'], stats['tail_5sigma']
print(f'train window {d0} -> {d1}: {stats["n_days"]} sessions x '
      f'{stats["n_assets"]} names ({stats["n_dead_by_end"]} dead by window end)')
print(f'EW daily: mean {stats["mean_daily"]:+.5f}, sd {stats["std_daily"]:.5f}, '
      f'skew {stats["skewness"]:.2f}, excess kurtosis {stats["excess_kurtosis"]:.2f}')
print(f'< -3 sigma: {t3["count"]} days vs {t3["normal_expected_days"]:.2f} '
      f'Normal-expected (x{t3["ratio"]:.1f})')
print(f'< -5 sigma: {t5["count"]} days vs {t5["normal_expected_days"]:.4f} '
      f'Normal-expected (x{t5["ratio"]:,.0f})')
for a, row in sorted(stats['cvar_by_level'].items(), reverse=True):
    print(f'CVaR_{a:g}: empirical {row["empirical"]*100:+.2f}%/day vs Normal '
          f'{row["normal"]*100:+.2f}%  (x{row["ratio"]:.2f})')
print(f'stress: {stats["n_stress_episodes"]} episodes hold all '
      f'{stats["n_stress_days"]} top-decile vol days; longest '
      f'{stats["longest_episode_days"]} sessions')
print(f'co-crash: calm {stats["co_crash_calm_mean"]:.1%} vs stress '
      f'{stats["co_crash_stress_mean"]:.1%} '
      f'(x{stats["co_crash_ratio"]:.1f}); worst day {stats["worst_day_co_crash"]:.0%}')

# Fail-loud pins on the headline EDA numbers quoted in the prose (drift = data change).
assert abs(stats['excess_kurtosis'] - 15.25) < 0.05, stats['excess_kurtosis']
assert t5['ratio'] > 1e3, 'the -5 sigma days should be >1000x the Normal expectation'
ratios = {a: v['ratio'] for a, v in stats['cvar_by_level'].items()}
assert ratios[0.25] < 1.0 < ratios[0.01], 'the CVaR curve should CROSS the Normal curve'
assert stats['co_crash_stress_mean'] > 3 * stats['co_crash_calm_mean']

fig = fig_stylised_facts(
    R, dates=dates, alive_mask=alive,
    footnote=(f'Descriptive EDA on the TRAIN window only (development top-30, '
              f'{d0} -> {d1}; anonymised ids; delisted names liquidate-to-cash) -- '
              'the sealed validation/test years are never read.'))
fig

### Reading the four panels

- **Heavy tails.** Excess kurtosis **15.2** (a Normal scores 0). Nine sessions land below $-5\sigma$ — a region where the matched Normal expects **0.0009 days** over the whole 12-year window, i.e. the observed count is **~10⁴×** the Gaussian expectation. Any reward that penalises "volatility" with a Gaussian mental model never prices these days.
- **The tail is a curve — and it *crosses* the Normal.** At the shallow fed level ($\alpha=0.25$) the empirical CVaR is *milder* than Normal-implied (ratio **≈0.84×**); at the deep level ($\alpha=0.01$) it is *far worse* (**≈1.66×**). Because the two curves **cross**, no single scalar — no one $\alpha$, no variance multiple — can represent the curve: any scalar summary is wrong in one direction at one end. This is precisely the information the six-coordinate fed vector transmits and a scalar collapses.
- **Clustering.** All 301 top-decile volatility days concentrate into just **19 episodes** (the longest — the GFC — runs 90 straight sessions); a time-averaged scalar cannot say *when* risk arrives, which is what a *conditional* tail measure is for.
- **Co-crashes.** On calm days **3.3%** of names sit below their own 5% tail (≈ independence); on stress days **19.8%** fall together — a **5.9×** amplification, and on the worst day *every* trading name breaches its own tail at once. Diversification fails exactly when the tail bites, so portfolio-level tail control cannot be delegated to breadth.

## 4 · What the channel carries — and why it may still go silent

The distributional arm is fed a six-coordinate profile of the *trained policy's own* realised left tail: the four CVaR levels marked in panel (b), plus a left-tail-mass and a robust-skew coordinate. The scalar arm is fed essentially one number. Section 3 showed the extra coordinates are *not redundant* on this panel. Why, then, pre-register a **null**?

**The numeracy bottleneck (the headline mechanism).** Even when the richer vector is supplied, frontier LLMs compare *close small floats* at only ~50–70% accuracy (arXiv:2602.07812; NUMCoT, arXiv:2406.02864). Two sibling candidates' CVaR-5% values typically differ in the **fourth decimal place** — squarely inside that failure regime. The channel can be *open* (the information is there) yet *silent* (the reader cannot act on it): a concrete, citable, falsifiable reason the effect should be ≈0 that is about **legibility, not capacity** — and §8's SQ3b instrument tests exactly that.

In [ ]:
# The REAL fed-style tail profile of this panel's EW portfolio (train window, daily):
profile = {f'cvar_{int(a*100):02d}': v['empirical']
           for a, v in sorted(stats['cvar_by_level'].items())}
print('four CVaR coordinates of the fed vector (train-window EW, signed daily):')
for k, v in profile.items():
    print(f'  {k:>10} : {v:+.4f}')
print('  (+ left_tail_mass and robust_skew complete the six-coordinate profile;\n'
      '   per-candidate values are measured on each POLICY\'s own returns)')
gap = abs(profile['cvar_05']) * 0.01
print(f'\nnumeracy regime: telling {profile["cvar_05"]:+.4f} from a sibling '
      f'{profile["cvar_05"] - gap:+.4f} is a ~{gap:.1e} gap between close small\n'
      'negatives -- the documented ~50-70% LLM comparison-failure regime (SQ3b, sec. 8).')

## 5 · Load the analysis bundle (synthetic NULL demo)

One call produces a NULL-shaped bundle with the exact schema the sealed-leg loader will emit — arms, per-seed scores by leg, contrasts, Bayes factors, MCS, and the mechanism arrays. **Post-campaign this is the only line that changes.**

In [ ]:
import make_figures as MF
from src.viz import figures as F

data = MF.synthesize_null(seed=RNG_SEED, n_seeds=30)  # <-- post-campaign: sealed-leg loader
print('arms          :', list(data['sharpe']))
print('legs          :', list(data['scores_by_leg']))
print('candidates    :', len(data['cand_arms']), 'authored reward programs (for the AST mechanism)')

## 6 · Headline H2 — co-primary equivalence (risk-adjusted **and** tail)

**Equivalence-first, by construction.** A bare $p>0.05$ says *"we saw nothing"*; a **TOST equivalence** says *"the effect is provably inside ±SESOI"* — a positive, falsifiable claim about a bounded effect. Each co-primary leg (H2-RA on Sharpe, H2-Tail on CVaR-5%) runs two one-sided tests against the pre-registered SESOI = 0.05; H2 as a whole is their **intersection–union**: it passes only if *every* leg passes, which is itself the multiplicity correction (Berger 1982). The forest draws a contrast **filled** iff its 90% TOST interval lies inside the ±SESOI band.

Three complementary readouts, one story:
1. the **TOST forest** — is the effect bounded inside the band?
2. **rliable IQM intervals** (Agarwal et al. 2021) — seed-level uncertainty, stratified-bootstrap, no seed-averaging;
3. **evidence for the null** — Bayes factors BF₀₁ and the Model Confidence Set: does the data *support* equivalence rather than merely fail to reject?

In [ ]:
SESOI = 0.05
for c in data['contrasts']:
    equiv = (c['tost_lo'] >= -SESOI) and (c['tost_hi'] <= SESOI)
    verdict = 'EQUIVALENT' if equiv else 'inconclusive'
    print(f"{c['leg']:>5} | {c['label']:<16} est={c['estimate']:+.3f} "
          f"TOST=[{c['tost_lo']:+.3f}, {c['tost_hi']:+.3f}]  ->  {verdict}")

In [ ]:
fig = F.equivalence_forest(data['contrasts'])  # F5/F6 — the bankable-null figure
fig

In [ ]:
fig = F.rliable_intervals(data['scores_by_leg'])  # IQM + stratified-bootstrap CIs
fig

In [ ]:
fig = F.evidence_for_null(data['bf01_by_leg'], data['mcs'])  # BF01 + Model Confidence Set
fig

## 7 · The reward-program taxonomy — what did the designers actually *write*? (real)

**The instrument.** Every authored program is reduced to its canonical **AST shape-set** (node *types* only — identifiers, constants and comments are invisible, so renaming a variable or re-tuning a coefficient cannot fake novelty). Pairwise Jaccard similarity over shape-sets + connected components at ≥ 0.6 = the program **kinds**; each kind gets a medoid exemplar and a label from construct prevalence (`src.inference.reward_taxonomy`). Report-only, deterministic, disjoint from the m=6 family.

**The question it answers.** Do different feedback arms author different *kinds* of programs — or the same kinds reshaped? This is the discriminative-validity check on the whole mechanism story: an instrument that cannot even separate a *template sampler* from an *LLM author* could not possibly detect feedback-driven structure. The numbers below are **real** — the frozen prototype archive (6 arms × ~40 candidates).

In [ ]:
import json
tax = json.loads((ROOT / 'outputs' / 'tables' / 'reward_taxonomy_prototype.json').read_text(encoding='utf-8'))
pooled, per_arm = tax['pooled'], tax['per_arm']
assert tax['status'] == 'ok'
assert (pooled['n_programs'], pooled['n_kinds'], pooled['n_singletons']) == (239, 157, 152)
print(f"pooled: {pooled['n_programs']} programs -> {pooled['n_kinds']} kinds "
      f"({pooled['n_singletons']} singletons) at Jaccard >= "
      f"{pooled['sim_threshold']:g}, AST depth {pooled['depth']}, "
      f"{pooled['n_unparseable']} unparseable")

SEARCH_ARMS = {'random_search', 'bayes_opt'}
rows = []
for arm, e in sorted(per_arm.items()):
    rows.append({'arm': arm, 'designer': 'search' if arm in SEARCH_ARMS else 'LLM',
                 'programs': e['n_programs'], 'kinds': e['n_kinds_present'],
                 'entropy_bits': round(e['entropy_bits'], 3),
                 'max_entropy_bits': round(float(np.log2(max(e['n_programs'], 1))), 3)})
    if arm in SEARCH_ARMS:  # a template sampler must collapse to ONE structural kind
        assert e['n_kinds_present'] == 1 and e['entropy_bits'] == 0.0, arm
    else:                   # an LLM arm should be near-fully idiosyncratic
        assert e['n_kinds_present'] >= 39, arm
composition = pd.DataFrame(rows).set_index('arm')

print('\nkinds shared by MORE than one program (everything else is a singleton):')
for k in pooled['kinds']:
    if k['size'] > 1:
        arms_of = sorted({m.split('/', 1)[0] for m in k['members']})
        print(f"  {k['kind_id']}: size {k['size']:>2}  arms={arms_of}  "
              f"label='{k['label']}'")

sens = tax['sensitivity']
print('\nthreshold sensitivity (is the taxonomy a threshold artifact?):')
print('  n_kinds:', {r['threshold']: r['n_kinds'] for r in sens['by_threshold']})
print('  adjacent Rand index:', {r['pair']: round(r['rand_index'], 4)
                                 for r in sens['adjacent_stability']})
composition

### Reading the taxonomy

- **The two search arms collapse to exactly one kind each** (entropy 0.0 bits): all 40 `random_search` programs are *one re-parameterised template*, and likewise `bayes_opt` — the AST shape-set sees through the constant-tuning entirely. That the instrument recovers this known ground truth is its **discriminative validity**.
- **The LLM arms are near-fully idiosyncratic**: 152 of the 157 pooled kinds are singletons, and every LLM arm's kind-entropy sits at ≈ its own maximum (log₂(40) ≈ 5.32 bits) — the LLM writes a structurally new program almost every call.
- **The only shared kinds cut *across* arms, not within them.** The three non-search multi-member kinds (`kind_003/004/005`) each span **two different feedback arms** (distributional–scalar_cvar5, distributional–placebo, scalar–scalar_cvar5). Structural twins appearing across conditions — including the *placebo* — is exactly what the **null** predicts: the fed signal is not organising program *structure* by arm.
- **Not a threshold artifact**: re-clustering at 0.5/0.6/0.7 keeps pair-agreement (Rand index) at 0.979 and 0.9998 between adjacent cuts.

Post-campaign, the identical induction re-runs on the 7-arm campaign archive (30 candidates/arm) via `scripts/build_taxonomy.py`.

## 8 · Mechanism — the originality kernel (a 3-link causal chain)

The deep contribution is *where* the channel acts — or fails to. The chain

$$\text{fed signal} \;\xrightarrow{\;a\;}\; \text{authored code} \;\xrightarrow{\;b\;}\; \text{policy} \;\longrightarrow\; \text{realised tail}$$

is decomposed into three sub-questions, each with a dedicated **report-only** instrument (disjoint from the frozen testing family):

| sub-question | what it asks, plainly | instrument | module |
|---|---|---|---|
| **SQ1 responsiveness** | when the *fed numbers* move, does the *code* move? | rank-correlation of fed-Δ vs reward-code-Δ, bootstrap CI | `src.inference.responsiveness` |
| **SQ2 transmission** | when the *code* moves, does the *outcome* move? | single-mediator decomposition, bootstrap CI on the indirect path $a\times b$ | `src.inference.mediation` |
| **SQ3 specificity** | is it *genuine use* of the signal or a surface echo? | AST-structural named-vs-blinded + the numeracy/legibility differential | `src.inference.contamination`, `responsiveness` |

**Why a null is informative here.** If SQ1 is null (path $a \approx 0$), the indirect effect $a\times b$ collapses **for any** downstream strength $b$ — the chain is severed at the *first* hop, and the performance equivalence of §6 is *explained* (the designer never routed the signal into code), not merely observed. The cells below run each instrument on **seeded synthetic nulls** so the logic is visible; post-campaign the same calls run on the archive.

In [ ]:
from src.inference.responsiveness import (
    responsiveness, legible_format_responsiveness_differential)
from src.inference.mediation import mediation_analysis
from src.inference.contamination import named_vs_blinded_structural
rng = np.random.default_rng(0)

# SQ1 — illustrative NULL: the authored-code feature does not track the fed tail signal.
n = 80
fed = rng.normal(size=n)
code_feat = rng.normal(size=n)            # independent of `fed` -> responsiveness ~ 0
sq1 = responsiveness(fed, code_feat, n_boot=800, rng=np.random.default_rng(1))
print('SQ1 responsiveness  : coef=%+.3f  CI=[%+.3f, %+.3f]  responsive=%s'
      % (sq1['coef'], sq1['ci_low'], sq1['ci_high'], sq1['responsive']))
assert not sq1['responsive']  # the seeded null must read as a null

In [ ]:
# SQ2 — mediation fed -> code -> outcome. With SQ1 null (path a ~ 0), the indirect
# effect a*b collapses even though the code->outcome link b is REAL by construction.
outcome = 0.9 * code_feat + 0.2 * rng.normal(size=n)
sq2 = mediation_analysis(fed, code_feat, outcome, n_boot=800,
                         rng=np.random.default_rng(2))
print('SQ2 mediation       : a=%+.3f  b=%+.3f  indirect(a*b)=%+.3f  '
      'CI=[%+.3f, %+.3f]  mediated=%s'
      % (sq2['a'], sq2['b'], sq2['indirect'], sq2['ci_low'], sq2['ci_high'],
         sq2['mediated']))
print('  -> the chain is severed at link 1 (fed -> code): a~0 => indirect~0 for ANY b.')

In [ ]:
# SQ3a — AST-structural named-vs-blinded: does revealing dataset identity change the
# program STRUCTURE? Blinding renames identifiers and re-tunes constants -- invisible to
# the AST shape-set, so structure locked to the DATA (not the name) scores as identical.
named = ['def reward(r):\n    return r.mean() - 0.5 * cvar(r)',
         'def reward(r):\n    return r.mean() / (r.std() + 1e-8)',
         'def reward(r):\n    return r.mean() - drawdown(r)']
blinded = ['def reward(x):\n    return x.mean() - 0.9 * cvar(x)',
           'def reward(z):\n    return z.mean() / (z.std() + 1e-8)',
           'def reward(w):\n    return w.mean() - drawdown(w)']
sq3 = named_vs_blinded_structural(named, blinded, rng=np.random.default_rng(3))
print('SQ3 AST-structural  : paired=%.3f  within-floor=%.3f  data_locked=%s'
      % (sq3['paired_mean'], sq3['within_blinded_mean'], sq3['data_locked']))
print('  -> identifier/constant renaming is invisible to the AST shape '
      '(structure is data-locked).')

In [ ]:
# SQ3b — the numeracy/legibility differential: does rendering the SAME tail content
# legibly RAISE responsiveness? A positive, zero-excluding gap => the bottleneck is
# LEGIBILITY (fixable by formatting), not model capacity -- the falsifiable scaling
# hypothesis behind the sec. 4 numeracy argument.
xl = rng.normal(size=120); ml = 0.85 * xl + 0.3 * rng.normal(size=120)  # legible: tracks
xr = rng.normal(size=120); mr = 0.05 * xr + rng.normal(size=120)        # raw: washes out
diff = legible_format_responsiveness_differential(
    xl, ml, xr, mr, n_boot=800, rng=np.random.default_rng(4))
print('SQ3 legibility diff : legible=%+.3f  raw=%+.3f  differential=%+.3f  '
      'CI=[%+.3f, %+.3f]  helps=%s'
      % (diff['coef_legible'], diff['coef_raw'], diff['differential'],
         diff['ci_low'], diff['ci_high'], diff['legibility_helps']))

In [ ]:
# The mechanism figures: the responsiveness scatter (F8b) and the 3-D reward-code
# embedding (clusters cutting ACROSS arms = the taxonomy's cross-arm twins, in 3-D).
fig = F.responsiveness_scatter(data['fed_delta'], data['reward_delta'], rho=data['rho'])
fig

In [ ]:
from src.viz import advanced as ADV
fig = ADV.reward_embedding_3d(data['ast_distance'], data['cand_arms'])
fig

## 9 · Robustness (post-campaign)

The confirmatory run re-estimates the headline under: the **delisting-treatment band** `{0, -30, -55, -100}%` (a sensitivity *surface*, never a single hidden choice — note the provenance walkthrough shows the corrected Shumway band-end `univ5s` is byte-identical to the headline panel, because every dead name's terminal return was already observed); **regime-stratified** tail metrics (calm vs crisis); the **PBO/CSCV** overfitting probability; and **FZ0 ES** backtests. Each is wired through `src/inference/` and renders in the same house style; the cells activate when the sealed-leg artifacts exist.

## 10 · Honest limitations & interpretation

- **The null is a boundary condition, not a non-result.** It says tail *specificity* adds no marginal value *over general risk-adjustment* **for a bounded, numerically bottlenecked authoring agent on this panel** — and the mechanism kernel locates *why* (SQ1/SQ3). It predicts the channel re-opens with a more legible rendering or a stronger numeric model: a falsifiable scaling hypothesis, not a shrug.
- **Endogeneity.** The fed tail is the trained policy's *own* realised returns, so H2 compares two coupled reward→policy→measurement loops; the estimator is critic-agnostic but **not** agent-independent. We never claim otherwise.
- **Power.** Tight equivalence bounds at 30 seeds are underpowered for some sub-tests (e.g. the named-vs-blinded TOST); we report effect sizes + CIs + achieved power, never bare non-rejections.
- **Generality.** Search width K, one model family for the confirmatory leg, one asset class, one panel. The pre-registered multi-model panel (ADR-039: Claude Opus + a strong open-weights coder) is the generality probe, secondary by design.

## 11 · Figure manifest & how to regenerate

Everything regenerates deterministically from the repo:

```bash
python scripts/make_figures.py --out outputs/figures          # headline + 3-D + GIF suite
python scripts/make_figures.py --out outputs/figures --no-advanced   # headline only
python scripts/build_taxonomy.py --root outputs/prototype     # the section-7 tables
python scripts/build_notebook_results.py                      # THIS notebook
python scripts/build_notebook_provenance.py                   # the provenance companion
```

Post-campaign, the same figure script loads the sealed-leg results (`--results-root`) and re-renders the identical suite from real data.

In [ ]:
print('figure entry points (scripts/make_figures.py):',
      [n for n in dir(MF) if n.startswith('render')])
print('headline figures  (src/viz/figures.py)  :', list(F.__all__))
print('advanced figures  (src/viz/advanced.py) :',
      [f for f in ADV.__all__ if f != 'classical_mds'])